In [16]:
# Install imbalanced-learn for SMOTE
import subprocess
subprocess.check_call(['.venv\\Scripts\\pip.exe', 'install', '-q', 'imbalanced-learn'])
print("SMOTE library installed!")

SMOTE library installed!


In [17]:
# IMPORT LIBRARIES

import sys, os, glob, gc, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb

from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, label_binarize
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    log_loss, roc_auc_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc as sk_auc
)
from imblearn.over_sampling import SMOTE
import joblib

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120

print("="*80)
print("RETRAINING WITH AGGRESSIVE SMOTE (50%) + HEAVY CLASS WEIGHTS")
print("="*80)

RETRAINING WITH AGGRESSIVE SMOTE (50%) + HEAVY CLASS WEIGHTS


In [18]:
# CONFIG

CSV_FOLDER   = './MachineLearningCVE'  
LABEL_COL    = 'Label'
N_FOLDS      = 10
TEST_SIZE    = 0.30        
RANDOM_STATE = 42
OUTPUT_DIR   = '.'
os.makedirs(OUTPUT_DIR, exist_ok=True)

import sklearn
print(f'Scikit-learn : {sklearn.__version__}')
print(f'LightGBM     : {lgb.__version__}')
print(f'Pandas       : {pd.__version__}')

Scikit-learn : 1.8.0
LightGBM     : 4.6.0
Pandas       : 3.0.3


In [19]:
# LOAD DATA

csv_files = sorted(glob.glob(os.path.join(CSV_FOLDER, '*.csv')))
if not csv_files:
    raise FileNotFoundError(f'No CSV files in {CSV_FOLDER}')

dfs = []
total_rows_read = 0
SAMPLE_ROWS = 2000000  # Reduced from 5M to fit SMOTE in memory

for f in csv_files:
    print(f'  Loading {os.path.basename(f)}...', end='')
    chunks = []
    rows_from_file = 0
    
    for chunk in pd.read_csv(f, chunksize=25000, low_memory=False, encoding='latin-1'):
        rows_from_file += len(chunk)
        total_rows_read += len(chunk)
        chunks.append(chunk)
        if total_rows_read >= SAMPLE_ROWS:
            break
    
    if chunks:
        file_df = pd.concat(chunks, ignore_index=True)
        dfs.append(file_df)
        print(f' OK - {len(file_df):,} rows')
    
    if total_rows_read >= SAMPLE_ROWS:
        print(f'  Total rows limit ({SAMPLE_ROWS:,}) reached')
        break

df = pd.concat(dfs, ignore_index=True)
print(f'\nTotal rows loaded: {len(df):,}')
del dfs, chunks; gc.collect()
print('OK')

  Loading Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv... OK - 225,745 rows
  Loading Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv... OK - 286,467 rows
  Loading Friday-WorkingHours-Morning.pcap_ISCX.csv... OK - 191,033 rows
  Loading Monday-WorkingHours.pcap_ISCX.csv... OK - 529,918 rows
  Loading Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv... OK - 288,602 rows
  Loading Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv... OK - 170,366 rows
  Loading Tuesday-WorkingHours.pcap_ISCX.csv... OK - 325,000 rows
  Total rows limit (2,000,000) reached

Total rows loaded: 2,017,131
OK


In [20]:
# CLEAN DATA

df.columns = df.columns.str.strip()

drop_cols = [c for c in ['Flow ID','Source IP','Destination IP','Timestamp',
                          'Src IP','Dst IP','src_ip','dst_ip'] if c in df.columns]
if drop_cols:
    df.drop(columns=drop_cols, inplace=True)

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(axis=1, thresh=int(len(df) * 0.5), inplace=True)  
df.dropna(axis=0, inplace=True)                               

before = len(df)
df.drop_duplicates(inplace=True)
print(f'Dedup removed {before - len(df):,} rows')
print(f'Shape after cleaning: {df.shape}')

Dedup removed 191,572 rows
Shape after cleaning: (1824054, 79)


In [21]:
# ENCODE & SCALE

le = LabelEncoder()
df['label_enc'] = le.fit_transform(df[LABEL_COL].astype(str))
class_names = le.classes_
n_classes   = len(class_names)
print(f'\nClasses ({n_classes})')

X = df.drop(columns=[LABEL_COL, 'label_enc']).select_dtypes(include=[np.number])
y = df['label_enc']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

scaler  = MinMaxScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test),      columns=X.columns)

del df; gc.collect()
print('OK')


Classes (10)
Train: (1276837, 78) | Test: (547217, 78)
OK


In [22]:
# AGGRESSIVE SMOTE: 15% OF MAJORITY CLASS (balanced oversampling)
print("\nApplying SMOTE (sampling_strategy=0.15)...")
print(f'Before SMOTE: {X_train_scaled.shape}')

# For multi-class SMOTE, convert float to dict: target = 15% of majority class
majority_count = np.bincount(y_train).max()
sampling_strategy_dict = {
    i: int(majority_count * 0.15) 
    for i in np.unique(y_train) 
    if np.bincount(y_train)[i] < majority_count
}
print(f'Sampling strategy (15% of majority): {sampling_strategy_dict}')

smote = SMOTE(sampling_strategy=sampling_strategy_dict, k_neighbors=5, random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
X_train_smote = pd.DataFrame(X_train_smote, columns=X.columns)

print(f'After SMOTE: {X_train_smote.shape}')

# Show class distribution change
print("\nClass distribution BEFORE SMOTE:")
print(y_train.value_counts().sort_index())
print("\nClass distribution AFTER SMOTE (15% oversampling):")
print(y_train_smote.value_counts().sort_index())


Applying SMOTE (sampling_strategy=0.15)...
Before SMOTE: (1276837, 78)
Sampling strategy (15% of majority): {np.int64(1): 167283, np.int64(2): 167283, np.int64(3): 167283, np.int64(4): 167283, np.int64(5): 167283, np.int64(6): 167283, np.int64(7): 167283, np.int64(8): 167283, np.int64(9): 167283}
After SMOTE: (2620767, 78)

Class distribution BEFORE SMOTE:
label_enc
0    1115220
1       1364
2      89610
3       4152
4         25
5      63486
6       1480
7       1029
8         15
9        456
Name: count, dtype: int64

Class distribution AFTER SMOTE (15% oversampling):
label_enc
0    1115220
1     167283
2     167283
3     167283
4     167283
5     167283
6     167283
7     167283
8     167283
9     167283
Name: count, dtype: int64


In [23]:
# COMPUTE HEAVY CLASS WEIGHTS
class_weights = compute_class_weight('balanced', classes=np.unique(y_train_smote), y=y_train_smote)
class_weight_dict = {i: w for i, w in enumerate(class_weights)}

print("\nClass weights (balanced):")
for cls_idx, weight in sorted(class_weight_dict.items()):
    cls_name = class_names[cls_idx]
    print(f"  {cls_name:30s}: {weight:.4f}")


Class weights (balanced):
  BENIGN                        : 0.2350
  Bot                           : 1.5667
  DDoS                          : 1.5667
  FTP-Patator                   : 1.5667
  Infiltration                  : 1.5667
  PortScan                      : 1.5667
  SSH-Patator                   : 1.5667
  Web Attack ï¿½ Brute Force    : 1.5667
  Web Attack ï¿½ Sql Injection  : 1.5667
  Web Attack ï¿½ XSS            : 1.5667


In [25]:
# TRAIN WITH 10-FOLD CROSS-VALIDATION - SAVE BEST FOLD
import time

print("\n" + "="*80)
print(f"TRAINING WITH {N_FOLDS}-FOLD STRATIFIED CROSS-VALIDATION")
print("="*80)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
fold_results = []
best_fold_score = 0
best_fold_models = None
best_fold_num = 0

fold_num = 0
start_time = time.time()

for train_idx, val_idx in skf.split(X_train_smote, y_train_smote):
    fold_num += 1
    fold_start = time.time()
    print(f"\n[FOLD {fold_num}/{N_FOLDS}]", end='')
    
    X_fold_train = X_train_smote.iloc[train_idx]
    y_fold_train = y_train_smote.iloc[train_idx]
    X_fold_val = X_train_smote.iloc[val_idx]
    y_fold_val = y_train_smote.iloc[val_idx]
    
    # Train base learners
    base_learners = {}
    
    # LightGBM (reduced to 50 trees for speed)
    lgbm_model = lgb.LGBMClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.7,
        num_leaves=31,
        class_weight='balanced',
        random_state=RANDOM_STATE,
        verbose=-1
    )
    lgbm_model.fit(X_fold_train, y_fold_train)
    base_learners['lgbm'] = lgbm_model
    lgbm_acc = lgbm_model.score(X_fold_val, y_fold_val)
    
    # Bagging (reduced to 15 trees for speed)
    bagging_model = BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=6),
        n_estimators=15,
        random_state=RANDOM_STATE,
        n_jobs=1
    )
    bagging_model.fit(X_fold_train, y_fold_train)
    base_learners['bagging'] = bagging_model
    bagging_acc = bagging_model.score(X_fold_val, y_fold_val)
    
    # Create meta-features for this fold
    meta_train = np.zeros((X_fold_train.shape[0], len(base_learners) * n_classes))
    meta_val = np.zeros((X_fold_val.shape[0], len(base_learners) * n_classes))
    
    for i, (name, model) in enumerate(base_learners.items()):
        meta_train[:, i*n_classes:(i+1)*n_classes] = model.predict_proba(X_fold_train)
        meta_val[:, i*n_classes:(i+1)*n_classes] = model.predict_proba(X_fold_val)
    
    # Train meta-learner
    meta_learner = LogisticRegression(
        max_iter=500,
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=1
    )
    meta_learner.fit(meta_train, y_fold_train)
    
    # Evaluate fold
    y_pred = meta_learner.predict(meta_val)
    fold_acc = accuracy_score(y_fold_val, y_pred)
    fold_f1 = f1_score(y_fold_val, y_pred, average='macro', zero_division=0)
    
    fold_time = time.time() - fold_start
    fold_results.append({
        'fold': fold_num,
        'accuracy': fold_acc,
        'f1_macro': fold_f1,
        'time': fold_time
    })
    
    elapsed = time.time() - start_time
    eta = (elapsed / fold_num) * (N_FOLDS - fold_num)
    print(f" - Acc: {fold_acc:.4f} | F1: {fold_f1:.4f} | Time: {fold_time:.1f}s | ETA: {eta:.0f}s")
    
    # Save best fold
    if fold_acc > best_fold_score:
        best_fold_score = fold_acc
        best_fold_models = {
            'lgbm': lgbm_model,
            'bagging': bagging_model,
            'meta_learner': meta_learner
        }
        best_fold_num = fold_num

print("\n" + "="*80)
print(f"BEST FOLD: #{best_fold_num} with Accuracy: {best_fold_score:.4f}")
print("="*80)

# Create summary
fold_df = pd.DataFrame(fold_results)
print("\nFold Summary:")
print(fold_df[['fold', 'accuracy', 'f1_macro', 'time']].to_string(index=False))
print(f"\nMean Accuracy: {fold_df['accuracy'].mean():.4f} (±{fold_df['accuracy'].std():.4f})")
print(f"Mean Macro F1: {fold_df['f1_macro'].mean():.4f} (±{fold_df['f1_macro'].std():.4f})")
print(f"Total time: {time.time() - start_time:.1f}s")


TRAINING WITH 10-FOLD STRATIFIED CROSS-VALIDATION

[FOLD 1/10]
  Training LightGBM...
    Val Accuracy: 0.9738
  Training Bagging (DecisionTree)...
    Val Accuracy: 0.9258
  Ensemble Accuracy: 0.9745 | Macro F1: 0.9604
  ✓ NEW BEST FOLD (Acc: 0.9745)

[FOLD 2/10]
  Training LightGBM...
    Val Accuracy: 0.9735
  Training Bagging (DecisionTree)...
    Val Accuracy: 0.9256
  Ensemble Accuracy: 0.9746 | Macro F1: 0.9606
  ✓ NEW BEST FOLD (Acc: 0.9746)

[FOLD 3/10]
  Training LightGBM...
    Val Accuracy: 0.9742
  Training Bagging (DecisionTree)...
    Val Accuracy: 0.9257
  Ensemble Accuracy: 0.9747 | Macro F1: 0.9607
  ✓ NEW BEST FOLD (Acc: 0.9747)

[FOLD 4/10]
  Training LightGBM...
    Val Accuracy: 0.9729
  Training Bagging (DecisionTree)...
    Val Accuracy: 0.9256
  Ensemble Accuracy: 0.9740 | Macro F1: 0.9596

[FOLD 5/10]
  Training LightGBM...
    Val Accuracy: 0.9750
  Training Bagging (DecisionTree)...
    Val Accuracy: 0.9260
  Ensemble Accuracy: 0.9762 | Macro F1: 0.9631
  ✓

In [26]:
# CREATE META-FEATURES FOR STACKING
print("\nGenerating meta-features...")

# Train set meta-features
meta_features_train = np.zeros((X_train_smote.shape[0], len(base_learners) * n_classes))
for i, (name, model) in enumerate(base_learners.items()):
    proba = model.predict_proba(X_train_smote)
    meta_features_train[:, i*n_classes:(i+1)*n_classes] = proba

# Test set meta-features
meta_features_test = np.zeros((X_test_scaled.shape[0], len(base_learners) * n_classes))
for i, (name, model) in enumerate(base_learners.items()):
    proba = model.predict_proba(X_test_scaled)
    meta_features_test[:, i*n_classes:(i+1)*n_classes] = proba

print(f"Meta-feature matrix: Train {meta_features_train.shape} | Test {meta_features_test.shape}")


Generating meta-features...
Meta-feature matrix: Train (2620767, 20) | Test (547217, 20)


In [33]:
# COMPREHENSIVE METRICS ON TEST SET (BEST FOLD MODELS)
print("\n" + "="*80)
print("COMPREHENSIVE EVALUATION METRICS")
print("="*80)

# Use best fold models to generate predictions on test set
print(f"\nUsing BEST FOLD #{best_fold_num} models for test evaluation...")

# Generate meta-features using ONLY base learners (lgbm + bagging)
base_learner_names = ['lgbm', 'bagging']
n_base_learners = len(base_learner_names)
meta_features_test_best = np.zeros((X_test_scaled.shape[0], n_base_learners * n_classes))

for i, name in enumerate(base_learner_names):
    model = best_fold_models[name]
    proba = model.predict_proba(X_test_scaled)
    meta_features_test_best[:, i*n_classes:(i+1)*n_classes] = proba
    print(f"  Generated meta-features for {name}: shape {proba.shape}")

print(f"Total meta-features shape: {meta_features_test_best.shape}")

# Get predictions from best fold meta-learner
y_pred_best = best_fold_models['meta_learner'].predict(meta_features_test_best)
y_pred_proba_best = best_fold_models['meta_learner'].predict_proba(meta_features_test_best)

# Calculate comprehensive metrics
accuracy = accuracy_score(y_test, y_pred_best)
precision_macro = precision_score(y_test, y_pred_best, average='macro', zero_division=0)
precision_weighted = precision_score(y_test, y_pred_best, average='weighted', zero_division=0)
recall_macro = recall_score(y_test, y_pred_best, average='macro', zero_division=0)
recall_weighted = recall_score(y_test, y_pred_best, average='weighted', zero_division=0)
f1_macro = f1_score(y_test, y_pred_best, average='macro', zero_division=0)
f1_weighted = f1_score(y_test, y_pred_best, average='weighted', zero_division=0)

# Calculate AUC-ROC (one-vs-rest for multi-class)
try:
    auc_roc_macro = roc_auc_score(label_binarize(y_test, classes=np.arange(n_classes)), 
                                   y_pred_proba_best, average='macro', multi_class='ovr')
    auc_roc_weighted = roc_auc_score(label_binarize(y_test, classes=np.arange(n_classes)), 
                                     y_pred_proba_best, average='weighted', multi_class='ovr')
except:
    auc_roc_macro = 0.0
    auc_roc_weighted = 0.0
    print("  (AUC-ROC calculation skipped due to class distribution)")

# Print overall metrics
print("\n" + "-"*80)
print("OVERALL METRICS")
print("-"*80)
print(f"Accuracy:              {accuracy:.4f}")
print(f"Precision (Macro):     {precision_macro:.4f}")
print(f"Precision (Weighted):  {precision_weighted:.4f}")
print(f"Recall (Macro):        {recall_macro:.4f}")
print(f"Recall (Weighted):     {recall_weighted:.4f}")
print(f"F1-Score (Macro):      {f1_macro:.4f}")
print(f"F1-Score (Weighted):   {f1_weighted:.4f}")
if auc_roc_macro > 0:
    print(f"AUC-ROC (Macro):       {auc_roc_macro:.4f}")
    print(f"AUC-ROC (Weighted):    {auc_roc_weighted:.4f}")

# Per-class metrics
print("\n" + "-"*80)
print("PER-CLASS METRICS")
print("-"*80)

precision_per_class = precision_score(y_test, y_pred_best, average=None, zero_division=0)
recall_per_class = recall_score(y_test, y_pred_best, average=None, zero_division=0)
f1_per_class = f1_score(y_test, y_pred_best, average=None, zero_division=0)

metrics_df = pd.DataFrame({
    'Class': class_names,
    'Precision': precision_per_class,
    'Recall': recall_per_class,
    'F1-Score': f1_per_class,
    'Support': np.bincount(y_test, minlength=n_classes)
})
print(metrics_df.to_string(index=False))

# Classification report
print("\n" + "-"*80)
print("CLASSIFICATION REPORT")
print("-"*80)
print(classification_report(y_test, y_pred_best, target_names=class_names, digits=4))

# Summary table
print("\n" + "-"*80)
print("METRICS SUMMARY TABLE")
print("-"*80)
summary_data = {
    'Metric': ['Accuracy', 'Precision (Macro)', 'Precision (Weighted)', 
               'Recall (Macro)', 'Recall (Weighted)', 'F1 (Macro)', 'F1 (Weighted)'],
    'Score': [accuracy, precision_macro, precision_weighted, 
              recall_macro, recall_weighted, f1_macro, f1_weighted]
}
if auc_roc_macro > 0:
    summary_data['Metric'].extend(['AUC-ROC (Macro)', 'AUC-ROC (Weighted)'])
    summary_data['Score'].extend([auc_roc_macro, auc_roc_weighted])

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))


COMPREHENSIVE EVALUATION METRICS

Using BEST FOLD #5 models for test evaluation...
  Generated meta-features for lgbm: shape (547217, 10)
  Generated meta-features for bagging: shape (547217, 10)
Total meta-features shape: (547217, 20)

--------------------------------------------------------------------------------
OVERALL METRICS
--------------------------------------------------------------------------------
Accuracy:              0.9980
Precision (Macro):     0.7784
Precision (Weighted):  0.9985
Recall (Macro):        0.9014
Recall (Weighted):     0.9980
F1-Score (Macro):      0.8125
F1-Score (Weighted):   0.9981
AUC-ROC (Macro):       0.9995
AUC-ROC (Weighted):    0.9999

--------------------------------------------------------------------------------
PER-CLASS METRICS
--------------------------------------------------------------------------------
                       Class  Precision   Recall  F1-Score  Support
                      BENIGN   0.999962 0.998186  0.999073   4779

In [34]:
# ROC CURVES PER CLASS
print("\nGenerating ROC curves...")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

# Binarize y_test for multi-class ROC
y_test_bin = label_binarize(y_test, classes=np.arange(n_classes))
y_pred_proba_bin = y_pred_proba_best

# Plot ROC curve per class
for i in range(n_classes):
    if i < 6:  # Only plot first 6 classes to fit in grid
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_pred_proba_bin[:, i])
        try:
            auc_score = roc_auc_score(y_test_bin[:, i], y_pred_proba_bin[:, i])
        except:
            auc_score = 0
        
        axes[i].plot(fpr, tpr, lw=2, label=f'AUC = {auc_score:.3f}')
        axes[i].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
        axes[i].set_xlabel('False Positive Rate')
        axes[i].set_ylabel('True Positive Rate')
        axes[i].set_title(f'ROC Curve: {class_names[i]}')
        axes[i].legend(loc='lower right')
        axes[i].grid(alpha=0.3)

# Hide unused subplots if n_classes < 6
for i in range(n_classes, 6):
    axes[i].set_visible(False)

plt.tight_layout()
plt.savefig('roc_curves_best_fold.png', dpi=120, bbox_inches='tight')
print("ROC curves saved to: roc_curves_best_fold.png")
plt.close()

# Confusion matrix
print("\nGenerating confusion matrix...")
cm = confusion_matrix(y_test, y_pred_best)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title('Confusion Matrix - Raw Counts', fontsize=12, fontweight='bold')

# Normalized
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
disp_norm = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=class_names)
disp_norm.plot(ax=axes[1], cmap='RdYlGn', values_format='.2%')
axes[1].set_title('Confusion Matrix - Normalized', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('confusion_matrix_best_fold.png', dpi=120, bbox_inches='tight')
print("Confusion matrix saved to: confusion_matrix_best_fold.png")
plt.close()

print("\nAll visualizations completed!")


Generating ROC curves...
ROC curves saved to: roc_curves_best_fold.png

Generating confusion matrix...
Confusion matrix saved to: confusion_matrix_best_fold.png

All visualizations completed!


In [35]:
# CLASSIFICATION REPORT VISUALIZATION
print("\nGenerating Classification Report visualizations...")

# Calculate per-class metrics
precision_per_class = precision_score(y_test, y_pred_best, average=None, zero_division=0)
recall_per_class = recall_score(y_test, y_pred_best, average=None, zero_division=0)
f1_per_class = f1_score(y_test, y_pred_best, average=None, zero_division=0)
support_per_class = np.bincount(y_test, minlength=n_classes)

# Create classification report dataframe
clf_report_df = pd.DataFrame({
    'Precision': precision_per_class,
    'Recall': recall_per_class,
    'F1-Score': f1_per_class,
    'Support': support_per_class
}, index=class_names)

# --- Figure 1: Per-Class Metrics Heatmap ---
fig, ax = plt.subplots(figsize=(12, 6))

# Create heatmap of metrics
metrics_data = clf_report_df[['Precision', 'Recall', 'F1-Score']].T
sns.heatmap(metrics_data, annot=True, fmt='.3f', cmap='RdYlGn', cbar_kws={'label': 'Score'},
            ax=ax, vmin=0, vmax=1, linewidths=0.5, linecolor='gray')
ax.set_title('Classification Report Heatmap - Per-Class Metrics', fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Classes', fontsize=12, fontweight='bold')
ax.set_ylabel('Metrics', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('classification_report_heatmap.png', dpi=120, bbox_inches='tight')
print("Classification report heatmap saved to: classification_report_heatmap.png")
plt.close()

# --- Figure 2: Grouped Bar Charts ---
fig, ax = plt.subplots(figsize=(16, 6))

x = np.arange(len(class_names))
width = 0.25

bars1 = ax.bar(x - width, precision_per_class, width, label='Precision', alpha=0.8, color='#2ecc71')
bars2 = ax.bar(x, recall_per_class, width, label='Recall', alpha=0.8, color='#3498db')
bars3 = ax.bar(x + width, f1_per_class, width, label='F1-Score', alpha=0.8, color='#e74c3c')

ax.set_xlabel('Classes', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Per-Class Performance Metrics (Precision, Recall, F1-Score)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.legend(fontsize=11, loc='lower right')
ax.set_ylim([0, 1.05])
ax.grid(axis='y', alpha=0.3, linestyle='--')

# Add value labels on bars
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.2f}',
                    ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('classification_report_bars.png', dpi=120, bbox_inches='tight')
print("Classification report bar chart saved to: classification_report_bars.png")
plt.close()

# --- Figure 3: Support Distribution ---
fig, ax = plt.subplots(figsize=(14, 6))

colors = plt.cm.Set3(np.linspace(0, 1, len(class_names)))
bars = ax.bar(class_names, support_per_class, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)

ax.set_xlabel('Classes', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Samples', fontsize=12, fontweight='bold')
ax.set_title('Test Set Distribution - Support per Class', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
plt.xticks(rotation=45, ha='right')

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('support_distribution.png', dpi=120, bbox_inches='tight')
print("Support distribution chart saved to: support_distribution.png")
plt.close()

# --- Figure 4: F1-Score vs Support Scatter Plot ---
fig, ax = plt.subplots(figsize=(12, 7))

scatter = ax.scatter(support_per_class, f1_per_class, s=300, c=range(len(class_names)), 
                     cmap='viridis', alpha=0.7, edgecolors='black', linewidth=2)

# Add class labels
for i, class_name in enumerate(class_names):
    ax.annotate(class_name, (support_per_class[i], f1_per_class[i]),
                xytext=(5, 5), textcoords='offset points', fontsize=9, fontweight='bold')

ax.set_xlabel('Support (Number of Test Samples)', fontsize=12, fontweight='bold')
ax.set_ylabel('F1-Score', fontsize=12, fontweight='bold')
ax.set_title('F1-Score vs Class Support', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='--')
ax.set_ylim([0, 1.05])

# Add colorbar
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Class Index', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('f1_score_vs_support.png', dpi=120, bbox_inches='tight')
print("F1-Score vs Support scatter plot saved to: f1_score_vs_support.png")
plt.close()

print("\n✓ All classification report visualizations completed!")
print("\nGenerated files:")
print("  - classification_report_heatmap.png")
print("  - classification_report_bars.png")
print("  - support_distribution.png")
print("  - f1_score_vs_support.png")


Generating Classification Report visualizations...
Classification report heatmap saved to: classification_report_heatmap.png
Classification report bar chart saved to: classification_report_bars.png
Support distribution chart saved to: support_distribution.png
F1-Score vs Support scatter plot saved to: f1_score_vs_support.png

✓ All classification report visualizations completed!

Generated files:
  - classification_report_heatmap.png
  - classification_report_bars.png
  - support_distribution.png
  - f1_score_vs_support.png


In [27]:
# TRAIN META-LEARNER (Logistic Regression with class weights)
print("\nTraining meta-learner with HEAVY class weights...")

meta_learner = LogisticRegression(
    max_iter=500,
    class_weight='balanced',  # Heavy balancing
    random_state=RANDOM_STATE,
    n_jobs=-1
)
meta_learner.fit(meta_features_train, y_train_smote)

print("Meta-learner trained OK")

# Evaluate ensemble
y_pred = meta_learner.predict(meta_features_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\nEnsemble Accuracy on Test Set: {accuracy:.4f}")


Training meta-learner with HEAVY class weights...
Meta-learner trained OK

Ensemble Accuracy on Test Set: 0.9981


In [28]:
# SAVE BEST FOLD MODELS
print("\nSaving best fold trained models...")

os.makedirs('saved_model', exist_ok=True)

# Save best fold models
joblib.dump(best_fold_models['lgbm'], 'saved_model/lgbm_model.pkl')
joblib.dump(best_fold_models['bagging'], 'saved_model/bagging_model.pkl')
joblib.dump(best_fold_models['meta_learner'], 'saved_model/meta_learner.pkl')

# Save preprocessing objects
joblib.dump(scaler, 'saved_model/scaler.pkl')
joblib.dump(le, 'saved_model/label_encoder.pkl')

# Save all fold results for reference
joblib.dump(fold_results, 'saved_model/fold_results.pkl')
joblib.dump(fold_df, 'saved_model/fold_summary.pkl')

print("  lgbm_model.pkl saved (from best fold)")
print("  bagging_model.pkl saved (from best fold)")
print("  meta_learner.pkl saved (from best fold)")
print("  scaler.pkl saved")
print("  label_encoder.pkl saved")
print("  fold_results.pkl saved")
print("  fold_summary.pkl saved")
print(f"\nBest fold: #{best_fold_num} with Accuracy: {best_fold_score:.4f}")
print("All models saved OK")


Saving best fold trained models...
  lgbm_model.pkl saved (from best fold)
  bagging_model.pkl saved (from best fold)
  meta_learner.pkl saved (from best fold)
  scaler.pkl saved
  label_encoder.pkl saved
  fold_results.pkl saved
  fold_summary.pkl saved

Best fold: #5 with Accuracy: 0.9762
All models saved OK


In [29]:
# DETAILED EVALUATION
print("\n" + "="*80)
print("DETAILED EVALUATION ON TEST SET")
print("="*80)

y_pred = meta_learner.predict(meta_features_test)
y_pred_proba = meta_learner.predict_proba(meta_features_test)

print(f"\nOverall Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Macro F1-Score: {f1_score(y_test, y_pred, average='macro'):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=class_names))

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix saved")


DETAILED EVALUATION ON TEST SET

Overall Accuracy: 0.9981
Macro F1-Score: 0.8137

Classification Report:
                              precision    recall  f1-score   support

                      BENIGN       1.00      1.00      1.00    477953
                         Bot       0.58      1.00      0.74       584
                        DDoS       1.00      1.00      1.00     38404
                 FTP-Patator       1.00      1.00      1.00      1779
                Infiltration       0.80      0.73      0.76        11
                    PortScan       0.99      1.00      0.99     27208
                 SSH-Patator       1.00      1.00      1.00       635
  Web Attack ï¿½ Brute Force       0.73      0.64      0.68       441
Web Attack ï¿½ Sql Injection       0.32      1.00      0.48         6
          Web Attack ï¿½ XSS       0.39      0.63      0.49       196

                    accuracy                           1.00    547217
                   macro avg       0.78      0.90   

In [30]:
# VISUALIZE CONFUSION MATRIX
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Raw counts
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title('Confusion Matrix - Raw Counts')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

# Normalized
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
disp_norm = ConfusionMatrixDisplay(confusion_matrix=cm_normalized, display_labels=class_names)
disp_norm.plot(ax=axes[1], cmap='RdYlGn', values_format='.1%')
axes[1].set_title('Confusion Matrix - Normalized (%)')
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('True Label')

plt.tight_layout()
plt.savefig('confusion_matrix_aggressive_smote.png', dpi=120, bbox_inches='tight')
print("Confusion matrix visualization saved")
plt.close()

Confusion matrix visualization saved


In [31]:
# SUMMARY
print("\n" + "="*80)
print("TRAINING SUMMARY")
print("="*80)
print(f"\nTraining approach: AGGRESSIVE SMOTE (50%) + Heavy Class Weights")
print(f"  - SMOTE oversampling: 50% of majority class (vs 10% before)")
print(f"  - Class weights: Balanced on all models")
print(f"  - Base learners: LightGBM + Bagging")
print(f"  - Meta-learner: Logistic Regression with balanced weights")
print(f"\nResults:")
print(f"  Overall Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"  Macro F1-Score: {f1_score(y_test, y_pred, average='macro'):.4f}")
print(f"  Test set size: {len(y_test):,} samples")
print(f"\nModels saved to ./saved_model/")
print(f"Next: Test on domain-aware synthetic data to validate generalization")


TRAINING SUMMARY

Training approach: AGGRESSIVE SMOTE (50%) + Heavy Class Weights
  - SMOTE oversampling: 50% of majority class (vs 10% before)
  - Class weights: Balanced on all models
  - Base learners: LightGBM + Bagging
  - Meta-learner: Logistic Regression with balanced weights

Results:
  Overall Accuracy: 0.9981
  Macro F1-Score: 0.8137
  Test set size: 547,217 samples

Models saved to ./saved_model/
Next: Test on domain-aware synthetic data to validate generalization
